## Stage 4.01 — setup and pinned OpenVLA-OFT policy preflight
Creates an isolated Python 3.10 environment and downloads the exact frozen checkpoint. The first run is slow and uses substantial disk space.

In [1]:
!rm -rf ~/venv-stage4-openvla

In [2]:
# 1. Check the Python version and executable path of the venv
!~/venv-stage4-openvla/bin/python3 -c "import sys; print('Venv Path:', sys.executable); print('Venv Version:', sys.version.split()[0])"

# 2. (Optional) List the packages installed in this specific venv
!~/venv-stage4-openvla/bin/pip3 list

/bin/bash: line 1: /home/jupyter-smtm-5bba/venv-stage4-openvla/bin/python3: No such file or directory
/bin/bash: line 1: /home/jupyter-smtm-5bba/venv-stage4-openvla/bin/pip3: No such file or directory


In [3]:
import json
import os
import shutil
import subprocess
from pathlib import Path

GPU = "3"

R = Path.home() / "async-vla-latency-bench"
P = Path.home() / "LIBERO-plus"
OFT = Path.home() / "openvla-oft"
LERO = Path.home() / "lerobot-stage4"
ENV = Path.home() / "venv-stage4-openvla"
PY = ENV / "bin/python"
OUT = Path.home() / "stage4"
OUT.mkdir(exist_ok=True)

OFT_SHA = "e4287e94541f459edc4feabc4e181f537cd569a8"
LEROBOT_SHA = "2aba372b4e217cc47db28e0f836859b20d1456c9"
CKPT = "moojink/openvla-7b-oft-finetuned-libero-spatial-object-goal-10"
REV = "13cdacd486c504e65408fc3c9e12fec9c5bf0382"

for path in (
  R,
  P,
  P / "libero/libero/assets",
):
  if not path.exists():
      raise SystemExit(f"STOP: missing prerequisite {path}")

line = subprocess.run(
  [
      "nvidia-smi",
      "-i",
      GPU,
      "--query-gpu="
      "name,memory.total,memory.used,utilization.gpu,driver_version",
      "--format=csv,noheader,nounits",
  ],
  capture_output=True,
  text=True,
  check=True,
).stdout.strip()

print(line)

name, total, used, util, driver = [
  value.strip() for value in line.split(",")
]

assert "A100" in name, f"STOP: expected A100, found {name}"

total_mib = int(total)
used_mib = int(used)
utilization = int(util)
free_mib = total_mib - used_mib

# GPU 4 currently has a small external allocation. Proceed only while at
# least 34 GB remains free and the external process is not actively computing.
if free_mib < 34000 or utilization >= 5:
  raise SystemExit(
      f"STOP: physical GPU {GPU} lacks sufficient free capacity: {line}; "
      f"free={free_mib} MiB"
  )

print(
  f"WARNING: proceeding on partially occupied physical GPU {GPU}; "
  f"used={used_mib} MiB, free={free_mib} MiB, "
  f"utilization={utilization}%"
)


def pinned_checkout(path, url, sha):
  if not path.exists():
      subprocess.run(
          ["git", "clone", url, str(path)],
          check=True,
      )

  actual = subprocess.run(
      ["git", "-C", str(path), "rev-parse", "HEAD"],
      capture_output=True,
      text=True,
      check=True,
  ).stdout.strip()

  if actual != sha:
      dirty = subprocess.run(
          ["git", "-C", str(path), "status", "--porcelain"],
          capture_output=True,
          text=True,
          check=True,
      ).stdout

      if dirty:
          raise SystemExit(
              f"STOP: {path} is modified; preserve it before checkout"
          )

      subprocess.run(
          ["git", "-C", str(path), "fetch", "origin", sha],
          check=True,
      )
      subprocess.run(
          ["git", "-C", str(path), "checkout", "--detach", sha],
          check=True,
      )

  resolved = subprocess.run(
      ["git", "-C", str(path), "rev-parse", "HEAD"],
      capture_output=True,
      text=True,
      check=True,
  ).stdout.strip()

  if resolved != sha:
      raise SystemExit(
          f"STOP: checkout mismatch for {path}: "
          f"expected {sha}, found {resolved}"
      )


pinned_checkout(
  OFT,
  "https://github.com/moojink/openvla-oft.git",
  OFT_SHA,
)

pinned_checkout(
  LERO,
  "https://github.com/huggingface/lerobot.git",
  LEROBOT_SHA,
)

if not PY.exists():
  # py310 = shutil.which("python3.10")
  py312 = shutil.which("python3.12")
  conda = shutil.which("conda") or shutil.which("mamba")
  
  # if py310:
  if py312:
      subprocess.run(
          #[py310, "-m", "venv", str(ENV)],
          [py312, "-m", "venv", str(ENV)],
          check=True,
      )
  elif conda:
      subprocess.run(
          [
              conda,
              "create",
              "-y",
              "-p",
              str(ENV),
              #"python=3.10.14",
              "python=3.12",
              "pip",
          ],
          check=True,
      )
  else:
      raise SystemExit(
          # "STOP: Python 3.10 or conda/mamba is required"
          "STOP: Python 3.12 or conda/mamba is required"
      )

# print("PASS: pinned checkouts and Python 3.10 environment ready")
print("PASS: pinned checkouts and Python 3.12 environment ready")

NVIDIA A100-SXM4-40GB, 40960, 12829, 100, 595.71.05


SystemExit: STOP: physical GPU 3 lacks sufficient free capacity: NVIDIA A100-SXM4-40GB, 40960, 12829, 100, 595.71.05; free=28131 MiB

/opt/tljh/user/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3783: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
'''
marker=ENV/".stage4_dependencies_complete"
if not marker.exists():
    subprocess.run([str(PY),"-m","pip","install","--upgrade","pip","setuptools","wheel"],check=True)
    subprocess.run([str(PY),"-m","pip","install","-e",str(OFT)],check=True)
    subprocess.run([str(PY),"-m","pip","install","--no-deps","-e",str(LERO)],check=True)
    subprocess.run([str(PY),"-m","pip","install","-e",str(R),"pandas","pyarrow","pytest","gymnasium","hf-libero"],check=True)
    marker.write_text("complete\n")
env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONPATH":str(OFT)+os.pathsep+str(R)})
versions=subprocess.run([str(PY),"-c","import sys,torch,transformers; print(sys.version.split()[0],torch.__version__,transformers.__version__)"],capture_output=True,text=True,check=True,env=env).stdout.strip(); print("runtime",versions)
pyver,torchver,transformersver=versions.split(); assert pyver.startswith("3.10.") and torchver.startswith("2.2.0") and transformersver.startswith("4.40.1")
subprocess.run([str(PY),"-m","pytest","-q",str(R/"async_vla_benchmark/tests")],cwd=R,env=env,check=True)
code=f"from huggingface_hub import snapshot_download; print(snapshot_download(repo_id='{CKPT}',revision='{REV}'))"
snapshot=subprocess.run([str(PY),"-c",code],env=env,capture_output=True,text=True,check=True).stdout.strip().splitlines()[-1]; SNAP=Path(snapshot); (OUT/"stage4_checkpoint_snapshot.txt").write_text(str(SNAP)+"\n")
required=["config.json","dataset_statistics.json","action_head--300000_checkpoint.pt","proprio_projector--300000_checkpoint.pt"]; missing=[x for x in required if not (SNAP/x).is_file()]
if missing: raise SystemExit(f"STOP: pinned checkpoint snapshot missing {missing}")
packages=subprocess.run([str(PY),"-m","pip","freeze"],capture_output=True,text=True,check=True).stdout.splitlines()
prov={"repository_sha":subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(),"libero_plus_sha":subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(),"lerobot_sha":LEROBOT_SHA,"openvla_oft_sha":OFT_SHA,"checkpoint_id":CKPT,"checkpoint_revision":REV,"checkpoint_snapshot":str(SNAP),"gpu_index":GPU,"gpu":line,"python":pyver,"packages":packages}
(OUT/"stage4_preflight_environment.json").write_text(json.dumps(prov,indent=2)+"\n"); print("PASS: Stage 4 pinned OpenVLA-OFT preflight complete")
'''

In [ ]:
marker=ENV/".stage4_dependencies_complete"
if not marker.exists():
    # --- FIX DEPENDENCY CONFLICTS FOR PYTHON 3.12 ---
    DLIMP = Path.home() / "dlimp_openvla"

    # 1. Clone dlimp locally if not already present
    if not DLIMP.exists():
        subprocess.run(["git", "clone", "https://github.com/moojink/dlimp_openvla", str(DLIMP)], check=True)

    # 2. Patch dependencies in local dlimp repository
    for fname in ["pyproject.toml", "setup.py", "requirements.txt"]:
        ffile = DLIMP / fname
        if ffile.exists():
            content = ffile.read_text()
            content = content.replace("tensorflow==2.15.0", "tensorflow>=2.16.1")
            content = content.replace('"tensorflow_graphics==2021.12.3",', "")
            content = content.replace("tensorflow_graphics==2021.12.3", "")
            content = content.replace('"tensorflow-graphics==2021.12.3",', "")
            content = content.replace("tensorflow-graphics==2021.12.3", "")
            ffile.write_text(content)

    # 3. Patch openvla-oft to use tensorflow>=2.16.1, local dlimp, and remove tensorflow-graphics
    for fname in ["pyproject.toml", "setup.py", "requirements.txt"]:
        ffile = OFT / fname
        if ffile.exists():
            content = ffile.read_text()
            content = content.replace("tensorflow==2.15.0", "tensorflow>=2.16.1")
            content = content.replace('"tensorflow_graphics==2021.12.3",', "")
            content = content.replace("tensorflow_graphics==2021.12.3", "")
            content = content.replace('"tensorflow-graphics==2021.12.3",', "")
            content = content.replace("tensorflow-graphics==2021.12.3", "")
            content = content.replace("git+https://github.com/moojink/dlimp_openvla", f"file://{DLIMP}")
            ffile.write_text(content)
    # --- NEW CODE: Update TensorFlow dependency for Python 3.12 compatibility ---
    oft_pyproject = OFT / "pyproject.toml"
    if oft_pyproject.exists():
        config_data = oft_pyproject.read_text()
        config_data = config_data.replace("tensorflow==2.15.0", "tensorflow>=2.16.1")
        oft_pyproject.write_text(config_data)
    # ----------------------------------------------------------------------------
    subprocess.run([str(PY),"-m","pip","install","--upgrade","pip","setuptools","wheel"],check=True)
    subprocess.run([str(PY),"-m","pip","install","-e",str(OFT)],check=True)
    subprocess.run([str(PY),"-m","pip","install","--no-deps","-e",str(LERO)],check=True)
    subprocess.run([str(PY),"-m","pip","install","-e",str(R),"pandas","pyarrow","pytest","gymnasium","hf-libero"],check=True)
    marker.write_text("complete\n")
env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONPATH":str(OFT)+os.pathsep+str(R)})
versions=subprocess.run([str(PY),"-c","import sys,torch,transformers; print(sys.version.split()[0],torch.__version__,transformers.__version__)"],capture_output=True,text=True,check=True,env=env).stdout.strip(); print("runtime",versions)
pyver,torchver,transformersver=versions.split(); assert pyver.startswith("3.12") and torchver.startswith("2.2.0") and transformersver.startswith("4.40.1")
# --- AUTO-INITIALIZE LIBERO CONFIG TO AVOID INTERACTIVE PROMPT ---
import yaml
libero_dir = Path.home() / ".libero"
libero_dir.mkdir(parents=True, exist_ok=True)
config_file = libero_dir / "config.yaml"
if not config_file.exists():
    try:
        from libero.libero import get_default_path_dict
        path_dict = get_default_path_dict()
    except Exception:
        libero_pkg = Path(PY.parent.parent) / "lib" / f"python{pyver[:4]}" / "site-packages" / "libero" / "libero"
        path_dict = {
            "benchmark_root_path": str(libero_pkg),
            "bddl_files_default_path": str(libero_pkg / "bddl_files"),
            "init_states_default_path": str(libero_pkg / "init_files"),
            "dataset_default_path": str(Path.home() / "libero_datasets"),
        }
    with open(config_file, "w") as f:
        yaml.dump(path_dict, f)
# --- CREATE ROBOSUITE MACROS_PRIVATE TO PREVENT IMPORT ERROR ---
try:
    import robosuite
    robosuite_dir = Path(robosuite.__file__).parent
    macros_private = robosuite_dir / "macros_private.py"
    if not macros_private.exists():
        macros_private.write_text("# Auto-generated private macros\n")
except Exception:
    pass
# -------------------------------------------------------------
subprocess.run([str(PY),"-m","pytest","-q","-s",str(R/"async_vla_benchmark/tests")],cwd=R,env=env,check=True)
code=f"from huggingface_hub import snapshot_download; print(snapshot_download(repo_id='{CKPT}',revision='{REV}'))"
snapshot=subprocess.run([str(PY),"-c",code],env=env,capture_output=True,text=True,check=True).stdout.strip().splitlines()[-1]; SNAP=Path(snapshot); (OUT/"stage4_checkpoint_snapshot.txt").write_text(str(SNAP)+"\n")
required=["config.json","dataset_statistics.json","action_head--300000_checkpoint.pt","proprio_projector--300000_checkpoint.pt"]; missing=[x for x in required if not (SNAP/x).is_file()]
if missing: raise SystemExit(f"STOP: pinned checkpoint snapshot missing {missing}")
packages=subprocess.run([str(PY),"-m","pip","freeze"],capture_output=True,text=True,check=True).stdout.splitlines()
prov={"repository_sha":subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(),"libero_plus_sha":subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(),"lerobot_sha":LEROBOT_SHA,"openvla_oft_sha":OFT_SHA,"checkpoint_id":CKPT,"checkpoint_revision":REV,"checkpoint_snapshot":str(SNAP),"gpu_index":GPU,"gpu":line,"python":pyver,"packages":packages}
(OUT/"stage4_preflight_environment.json").write_text(json.dumps(prov,indent=2)+"\n"); print("PASS: Stage 4 pinned OpenVLA-OFT preflight complete")

In [ ]:
# List available Python versions
!ls /usr/bin/python*
shutil.which("python3.12")
shutil.which("python3.13")

# which python3.12
# which python3.13

In [ ]:
# 1. Check the Python version and executable path of the venv
!~/venv-stage4-openvla/bin/python3 -c "import sys; print('Venv Path:', sys.executable); print('Venv Version:', sys.version.split()[0])"

# 2. (Optional) List the packages installed in this specific venv
!~/venv-stage4-openvla/bin/pip3 list